# Production ML Pipeline

This notebook integrates everything:

1. Data understanding
2. Preprocessing
3. Dimensionality reduction (PCA)
4. Clustering (structure discovery)
5. Classification
6. Cross-validation
7. Hyperparameter optimization
8. Robust evaluation

Goal:

Build a reliable, unbiased, reproducible ML system.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris

data = load_iris()

X = data.data
y = data.target

print("Shape:", X.shape)
print("Features:", data.feature_names)

In [ ]:
# PIPELINE (NO DATA LEAKAGE is what we want here)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    
    # Scale features
    # WHY: ensure equal influence of all features
    ("scaler", StandardScaler()),
    
    # Reduce dimensionality
    # WHY: remove noise + simplify structure
    ("pca", PCA()),
    
    # Classification model
    # WHY: predict target
    ("clf", LogisticRegression())
])

In [ ]:
# HYPERPARAMETERS TO OPTIMIZE

param_grid = {

    # PCA dimensionality
    # WHY: tradeoff information vs simplicity
    "pca__n_components": [2, 3, 4],

    # Regularization strength
    # WHY: control overfitting
    "clf__C": [0.01, 0.1, 1, 10],

    # Optimization algorithm
    # WHY: affects convergence performance
    "clf__solver": ["lbfgs", "newton-cg"],

    # Max iterations
    # WHY: ensure convergence
    "clf__max_iter": [100, 500, 1000, 2000]
}

In [ ]:
# NESTED CROSS-VALIDATION

from sklearn.model_selection import GridSearchCV, cross_val_score, KFold

# INNER LOOP → hyperparameter tuning
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=inner_cv,
    scoring="accuracy"
)

# OUTER LOOP → unbiased evaluation
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    grid,
    X,
    y,
    cv=outer_cv
)

print("Nested CV scores:", scores)
print("Mean score:", scores.mean())
print("Std:", scores.std())


In [ ]:
# Train on full dataset with best parameters

grid.fit(X, y)

best_model = grid.best_estimator_

print("Best parameters:", grid.best_params_)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

preds = best_model.predict(X)

print("Accuracy:", accuracy_score(y, preds))
print("Precision:", precision_score(y, preds, average="weighted"))
print("Recall:", recall_score(y, preds, average="weighted"))
print("F1:", f1_score(y, preds, average="weighted"))

print("\nConfusion Matrix:")
print(confusion_matrix(y, preds))
